# Step 2. The trial before

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter/blob/main/tutorial/notebooks/2_sequence.ipynb)

A backup for the tutorial: the same step without an assistant. Run the cells from top to bottom.

In [ ]:
%pip install -q sweetpea pandas

**The question.** Does a conflict change how the next trial is handled?

**Prediction (conflict adaptation).** The congruency effect is smaller after an incongruent trial than after a congruent one.

**The alternative to rule out.** Nothing is adjusted; trials that repeat a colour or a word from the trial before are simply faster, and such repeats are spread unevenly.

**The analysis.** The congruency effect after a congruent trial against the congruency effect after an incongruent trial.

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, Exclude, MinimumTrials,
                      CrossBlock, synthesize_trials, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says

In [ ]:
# a derived factor: it follows from colour and word on the same trial
def same(colour, word):
    return colour == word


def different(colour, word):
    return colour != word


congruency = Factor("congruency", [
    DerivedLevel("congruent", WithinTrial(same, [colour, word])),
    DerivedLevel("incongruent", WithinTrial(different, [colour, word])),
])

### The trial before as a factor

A transition factor follows from the trial before: `[0]` is this trial, `[-1]` the one before.

In [ ]:
def after_congruent(congruency):
    return congruency[-1] == "congruent"


def after_incongruent(congruency):
    return congruency[-1] == "incongruent"


trial_before = Factor("trial_before", [
    DerivedLevel("congruent", Transition(after_congruent, [congruency])),
    DerivedLevel("incongruent", Transition(after_incongruent, [congruency])),
])

### The block

Colour, congruency and the trial before are crossed: 3 x 2 x 2 = 12 combinations. Twice is 24 trials, plus a first trial that has no trial before.

In [ ]:
block = CrossBlock(design=[colour, word, congruency, trial_before],
                   crossing=[colour, congruency, trial_before],
                   constraints=[MinimumTrials(25)])
trials = pd.DataFrame(synthesize_trials(block, 1, CMSGen)[0])
rest = trials.iloc[1:]                  # without the first trial
print(len(trials), "trials")
print(pd.crosstab(rest["trial_before"], rest["congruency"]))

### What else repeats?

Not in the design: does the ink colour, and so the response, repeat? Count it in each of the four combinations.

In [ ]:
trials["response_repeats"] = trials["colour"].eq(trials["colour"].shift())
rest = trials.iloc[1:]
print(pd.crosstab([rest["trial_before"], rest["congruency"]], rest["response_repeats"]))

**Check.** Are the four combinations equal? Is the response repeat equal across them? If not, the alternative explanation is still open: whatever is not in the crossing is left to chance.

**Try.** Run the block cell again a few times. Which counts stay the same, which change?